# 02 · Vitals simulator: giving every patient a heartbeat

**Who this is for:** you, after notebook 01.
**You should already know:** numpy arrays, pandas, simple matplotlib plots.

**By the end you will be able to:**
- explain the difference between **physiology** (what the body does) and **measurement** (what the device reports)
- build realistic vital signs **layer by layer**: personal baseline → slow wander → day/night rhythm → fever coupling → noise and movement artefacts → sensor faults
- push a patient's physiology with an **offset** (the trick the truth generator and drift scenarios use)
- simulate the whole hospital and sanity-check the result

**Outline**
1. Setup: load the inpatients from notebook 01
2. Physiology vs measurement
3. Layer 0: personal baselines
4. Layer 1: slow random wander
5. Layer 2: day/night rhythm
6. Layer 3: fever coupling (and offsets)
7. Layer 4: device noise and movement artefacts
8. Layer 5: sensor faults
9. The whole hospital
10. A miscalibrated sensor: measurement offsets
11. Exercise
12. Pitfalls and next steps

Code used: `generator/vitals.py` (tests in `tests/test_vitals.py`).

## 1. Setup: load the inpatients from notebook 01

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from generator import hospital, synthea, vitals


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found")


ROOT = find_repo_root()
RUN = synthea.SyntheaRun()
csv_dir = synthea.run_synthea(RUN, ROOT / "data" / "synthea" / f"p{RUN.population}_s{RUN.seed}")
profiles = synthea.build_profiles(
    synthea.load_tables(csv_dir), pd.Timestamp(RUN.reference_date, tz="UTC")
)
inpatients = hospital.admit_inpatients(profiles)
START = hospital.SIM_START
print(len(inpatients), "inpatients | simulated clock starts", START)

We'll follow one patient closely: the **first COPD patient on a respiratory unit**.

In [ ]:
star = inpatients[inpatients["copd"] & inpatients["unit_id"].str.endswith("RESPIRATORY")].iloc[[0]]
STAR = star.index[0]  # row position used to pick this patient out of the arrays
star[["device_id", "unit_id", "age", "sex", "copd", "hypertension", "on_beta_blocker"]]

In [ ]:
VITAL_LABELS = {
    "heart_rate": "Heart rate (bpm)",
    "resp_rate": "Breathing (/min)",
    "spo2": "SpO2 (%)",
    "temp_c": "Temperature (°C)",
    "sbp": "Systolic BP (mmHg)",
}


def plot_patient(series_by_label, timestamps, title, vitals_to_show=VITAL_LABELS):
    """One small chart per vital; each label in `series_by_label` is a dict vital -> 1-D array."""
    fig, axes = plt.subplots(
        len(vitals_to_show), 1, figsize=(11, 1.7 * len(vitals_to_show)), sharex=True
    )
    for ax, (v, label) in zip(axes, vitals_to_show.items(), strict=True):
        for name, series in series_by_label.items():
            ax.plot(timestamps, series[v], lw=1, label=name)
        ax.set_ylabel(label, fontsize=8)
    axes[0].legend(fontsize=8, loc="upper right")
    fig.suptitle(title)
    plt.tight_layout()


def physiology_for(config, hours=24, offsets=None, seed=7):
    """True vitals for every inpatient under `config`: (timestamps, baselines, arrays)."""
    ts = vitals.make_timestamps(START, hours)
    base_rng, phys_rng = np.random.default_rng(seed).spawn(2)
    baselines = vitals.make_baselines(inpatients, base_rng)
    return ts, baselines, vitals.simulate_physiology(baselines, ts, phys_rng, config, offsets)

## 2. Physiology vs measurement

The simulator keeps two layers apart, and this is the single most important idea in this notebook:

| Layer | What it is | Who uses it |
|---|---|---|
| **Physiology** | What the patient's body is *really* doing | The **truth generator** (notebook 03) decides who deteriorates from this |
| **Measurement** | What the monitor *reports*: physiology plus noise, artefacts, faults | The **data platform and model**. They never see the true physiology. |

Real hospitals work the same way: you only ever see the measurements. Keeping them separate lets us simulate a **broken sensor** (measurement changes, the patient doesn't), which is drift scenario 1.

We'll build the layers one at a time using `VitalsConfig` switches, starting with everything off:

In [ ]:
OFF = vitals.VitalsConfig(
    wander=False, circadian=False, fever_coupling=False, noise=False, artefacts=False, faults=False
)
OFF

## 3. Layer 0: personal baselines

Every patient gets their own "normal" values: drawn around typical adult values, then adjusted for **age, illnesses and medicines**.

In [ ]:
pd.DataFrame(vitals.POPULATION, index=["typical mean", "spread between people"]).T

| Adjustment | Effect |
|---|---|
| Age over 50 | Systolic BP rises 0.4 mmHg/year (stiffer arteries); heart rate slightly lower |
| COPD | SpO2 normal drops to ~91%; breathing +3/min |
| Heart failure / atrial fibrillation | Heart rate +8 / +10; AF also makes it more irregular |
| Beta-blocker | Heart rate −10 |
| Hypertension | Systolic +14, diastolic +8 |
| Diabetes or kidney disease | Systolic +5 |

Finally each baseline is kept within a plausible **resting** range (`vitals.RESTING_RANGE`), so nobody gets a resting heart rate of 30.

In [ ]:
ts, baselines, phys = physiology_for(OFF)
groups = {
    "COPD": inpatients["copd"].to_numpy(),
    "no COPD": ~inpatients["copd"].to_numpy(),
}
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for name, mask in groups.items():
    axes[0].hist(baselines.loc[mask, "spo2"], bins=20, alpha=0.6, label=name, density=True)
axes[0].set_title("Baseline SpO2")
axes[0].legend()
for name, mask in {
    "beta-blocker": inpatients["on_beta_blocker"].to_numpy(),
    "none": ~inpatients["on_beta_blocker"].to_numpy(),
}.items():
    axes[1].hist(baselines.loc[mask, "heart_rate"], bins=20, alpha=0.6, label=name, density=True)
axes[1].set_title("Baseline heart rate")
axes[1].legend()
plt.tight_layout()

**Interpretation:** COPD patients sit around 91% oxygen (normal for them). Patients on beta-blockers have visibly lower heart rates.
With every other layer off, our star patient's vitals are **flat lines** at their baseline:

In [ ]:
plot_patient(
    {"baseline only": {v: phys[v][STAR] for v in vitals.VITALS}}, ts, "Layer 0: baseline only"
)

## 4. Layer 1: slow random wander

Real vitals drift up and down over minutes to hours. We use a **mean-reverting random walk** (an *Ornstein–Uhlenbeck* process): each step keeps most of the previous deviation, adds a little randomness, and is gently pulled back toward the baseline.

`vitals.WANDER` sets two numbers per vital: the **typical size** of the wander, and the **memory** (how many minutes it takes to forget a deviation). Temperature changes slowly (180 min), breathing quickly (60 min).

In [ ]:
pd.DataFrame(vitals.WANDER, index=["typical size", "memory (min)"]).T

In [ ]:
ts, _, phys_wander = physiology_for(replace(OFF, wander=True))
plot_patient(
    {
        "baseline": {v: phys[v][STAR] for v in vitals.VITALS},
        "with wander": {v: phys_wander[v][STAR] for v in vitals.VITALS},
    },
    ts,
    "Layer 1: slow random wander",
)

## 5. Layer 2: day/night rhythm

Heart rate, blood pressure and temperature are higher in the late afternoon (~16:00) and lowest in the early morning (~04:00). This is the **circadian rhythm**.

In [ ]:
ts, _, phys_rhythm = physiology_for(replace(OFF, wander=True, circadian=True), hours=48)
_, _, phys_wander48 = physiology_for(replace(OFF, wander=True), hours=48)
plot_patient(
    {
        "wander": {v: phys_wander48[v][STAR] for v in vitals.VITALS},
        "wander + rhythm": {v: phys_rhythm[v][STAR] for v in vitals.VITALS},
    },
    ts,
    "Layer 2: day/night rhythm (48 hours)",
)

## 6. Layer 3: fever coupling, and offsets

Vitals aren't independent. When temperature rises, heart rate rises about **10 bpm per °C** and breathing about **2/min per °C** (`vitals.FEVER_COUPLING`).

To see it, we give everyone a fever using an **offset**: an array added to the physiology. Here +1.5 °C between hours 10 and 16.
Offsets are how the **truth generator** will make deteriorating patients get gradually worse, and how drift scenarios change the hospital.

In [ ]:
config_coupled = replace(OFF, wander=True, circadian=True, fever_coupling=True)
ts = vitals.make_timestamps(START, 24)
fever = np.zeros((len(inpatients), len(ts)))
hours_in = (ts - START) / pd.Timedelta(hours=1)
fever[:, (hours_in >= 10) & (hours_in < 16)] = 1.5

_, _, phys_calm = physiology_for(config_coupled)
_, _, phys_fever = physiology_for(config_coupled, offsets={"temp_c": fever})
plot_patient(
    {
        "no fever": {v: phys_calm[v][STAR] for v in vitals.VITALS},
        "fever 10:00-16:00": {v: phys_fever[v][STAR] for v in vitals.VITALS},
    },
    ts,
    "Layer 3: fever pushes heart rate and breathing up",
)

In [ ]:
window = (hours_in >= 10) & (hours_in < 16)
for v in ["temp_c", "heart_rate", "resp_rate"]:
    change = (phys_fever[v][:, window] - phys_calm[v][:, window]).mean()
    print(f"{v:<11} average change during fever: {change:+.1f}")

## 7. Layer 4: device noise and movement artefacts

Now the **measurement** layer. Two things corrupt readings:
- **noise:** every reading is slightly off (`vitals.NOISE_SD`), e.g. ±1.5 bpm, ±0.5% SpO2
- **movement artefacts:** when the patient moves a lot (motion > 0.6), optical sensors get confused. Heart rate jumps up and SpO2 reads falsely low.

Motion is low at night and higher during the day, with occasional bursts.

In [ ]:
cfg_noise = replace(
    OFF, wander=True, circadian=True, fever_coupling=True, noise=True, artefacts=True
)
res = vitals.simulate_vitals(inpatients, START, hours=24, seed=7, config=cfg_noise)
star_device = inpatients.loc[STAR, "device_id"]
star_readings = res.readings[res.readings["device_id"] == star_device].set_index("ts")

fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
axes[0].plot(res.timestamps, res.physiology["heart_rate"][STAR], label="true", lw=1.5)
axes[0].plot(star_readings.index, star_readings["heart_rate"], ".", ms=3, label="measured")
axes[0].set_ylabel("Heart rate")
axes[0].legend(fontsize=8)
axes[1].plot(res.timestamps, res.physiology["spo2"][STAR], lw=1.5)
axes[1].plot(star_readings.index, star_readings["spo2"], ".", ms=3)
axes[1].set_ylabel("SpO2")
axes[2].plot(res.timestamps, res.motion[STAR], color="grey")
axes[2].axhline(0.6, ls="--", color="red", lw=1)
axes[2].set_ylabel("Motion")
fig.suptitle("Layer 4: measured (dots) vs true (line). Spikes line up with motion above 0.6")
plt.tight_layout()

**Interpretation:** most dots hug the true line (noise). Occasional outliers (high heart rate, low SpO2) coincide with **motion bursts**. These are artefacts, not real health changes.
A good model and good data cleaning must cope with them. That's why the features use hourly **medians/minimums** and why Silver flags impossible values.

## 8. Layer 5: sensor faults

Real monitoring data has gaps and glitches (`VitalsConfig` defaults):
- **dropouts:** about 1.5% of messages are simply lost
- **stuck values:** occasionally a sensor freezes and repeats the same number for 30–60 minutes
- **battery outages:** batteries drain 1–2% per hour. When one dies, the device is **offline for 30–90 minutes** until it's swapped.

In [ ]:
full = vitals.simulate_vitals(inpatients, START, hours=24 * 3, seed=7)  # all layers on (defaults)
expected = len(inpatients) * len(full.timestamps)
lost = 1 - len(full.readings) / expected
print(f"messages expected: {expected:,} | received: {len(full.readings):,} | lost: {lost:.1%}")

In [ ]:
per_device = full.readings.groupby("device_id")["ts"].agg(["count", "min", "max"])
worst = per_device["count"].idxmin()
d = full.readings[full.readings["device_id"] == worst].set_index("ts")

fig, axes = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
axes[0].plot(d.index, d["battery_pct"], ".", ms=2)
axes[0].set_ylabel("Battery %")
axes[1].plot(d.index, d["heart_rate"], ".", ms=2)
axes[1].set_ylabel("Heart rate")
fig.suptitle(f"{worst}: the device with the most missing data. Gaps = dead battery being swapped")
plt.tight_layout()

## 9. The whole hospital

Everything on, all ~300 inpatients, 1 simulated day. This is what lands in **Bronze** (as JSON files) in Phase 1. Note it takes about a second.

In [ ]:
%%time
day = vitals.simulate_vitals(inpatients, START, hours=24, seed=42)
print(f"{len(day.readings):,} readings")

In [ ]:
day.readings.iloc[0].to_dict()

Notice the message has a **`device_id` but no `patient_id`**. Real monitors don't know who they're attached to.
The data platform links readings to patients through the **device assignments** table (from Azure SQL), in the Silver layer.

A quick realism check, comparing the middle 98% of readings with typical ward ranges:

In [ ]:
check = day.readings[list(vitals.VITALS)].quantile([0.01, 0.5, 0.99]).T.round(1)
check.columns = ["1st pct", "median", "99th pct"]
check["typical ward range"] = ["50-110", "12-24", "88-100", "36.0-37.8", "95-180", "55-105"]
check

## 10. A miscalibrated sensor: measurement offsets

`measurement_offsets` change **only what devices report**. Below, every Site B device reads SpO2 **3 points too low**. That's drift scenario 1 (a firmware bug).

In [ ]:
site_b = inpatients["site_id"].eq("SITE_B").to_numpy()
ts = vitals.make_timestamps(START, 24)
bias = np.zeros((len(inpatients), len(ts)))
bias[site_b, :] = -3

normal = vitals.simulate_vitals(inpatients, START, hours=24, seed=42)
faulty = vitals.simulate_vitals(
    inpatients, START, hours=24, seed=42, measurement_offsets={"spo2": bias}
)

b_devices = set(inpatients.loc[site_b, "device_id"])


def site_b_mean(result, layer):
    if layer == "true":
        return result.physiology["spo2"][site_b].mean()
    r = result.readings
    return r.loc[r["device_id"].isin(b_devices), "spo2"].mean()


pd.DataFrame(
    {
        "true SpO2 (body)": [site_b_mean(normal, "true"), site_b_mean(faulty, "true")],
        "measured SpO2 (device)": [
            site_b_mean(normal, "measured"),
            site_b_mean(faulty, "measured"),
        ],
    },
    index=["normal sensors", "faulty sensors"],
).round(2)

**Interpretation:** the patients' **true** oxygen is identical in both runs. Only the **measurements** dropped.
So when the monitoring alert fires on day 3, the correct response is to fix the sensors, **not to retrain the model**. A retrained model would learn the bug.

## 11. Exercise: how often does SpO2 fall below 92%?

Clinicians watch for SpO2 **below 92%**. Using `day.readings` (all layers on):
1. Join the readings to `inpatients` on `device_id` to get each reading's `copd` flag.
2. Compute the % of readings below 92 for COPD vs non-COPD patients.
3. Think: why might an SpO2 alarm at 92% be a bad idea for COPD patients? (Hint: NEWS2 has a separate oxygen scale for them.)

In [ ]:
# Try it yourself
joined = None  # readings + copd flag

low_spo2_pct = None  # % below 92, by copd
low_spo2_pct

In [ ]:
# Answer (run after trying)
joined = day.readings.merge(inpatients[["device_id", "copd"]], on="device_id")
low_spo2_pct = joined.groupby("copd")["spo2"].apply(lambda s: round(100 * (s < 92).mean(), 1))
low_spo2_pct.rename("% readings below 92")

COPD patients spend a large share of time below 92%: that's their normal. A single 92% alarm would ring constantly for them (alarm fatigue).
NEWS2 handles this with a separate **SpO2 scale 2** for patients with chronic lung disease. Our model gets the `copd` flag as a feature for the same reason.

## 12. Pitfalls and next steps

**Pitfall: treating measurements as truth.** Readings contain noise, artefacts and frozen values. Always aggregate (medians, minimums over an hour) and validate ranges before modelling.

**Pitfall: silently dropping missing data.** About 3% of messages never arrive, sometimes in long gaps. Count missingness and keep it as a feature (*"missing readings in last hour"*); gaps can themselves be informative.

**Pitfall: forgetting physiology vs measurement in drift work.** A sensor fault changes the data but not the patients. Retraining on it bakes the bug into the model.

**Optional extension:** set `vitals.VitalsConfig(dropout_prob=0.10)` and see how many messages remain, then think about how hourly features would cope.

**Next notebook:** `03_truth_generator_and_trajectories`. We decide which patients deteriorate, make their vitals worsen 2–12 hours beforehand (using offsets), and write the delayed outcome events.